<div class="jcs-title"><div class="jcs-kicker">Human–AI Commons · JCS · 03</div>
<h1>Where sources empty, and what local interventions change</h1>
<p>The load on each source and the three interventions of the paper, on one of its worlds.</p></div>

The paper finds that the sources which stay nearly empty are those from which agents request more than they renew, and that no change of the payoffs alters this. It then applies three interventions to the AI agents: **scarcity pricing**, an **extraction cap** and **capacity-based siting**. This notebook follows both steps on world 0 of the paper's interventions experiment, with the paper's own configuration, `configs/local_measures.json`. The numbers printed for this world are those the experiment saves; the paper reports averages over 64 such worlds.

Each simulation takes a few seconds. No result bundle is needed; the last section also reads the experiment's tables when they have been generated.

**Before opening:** complete the [JCS quick start](../../README.md#jcs-quick-start) once. From the repository root, launch `.venv/bin/jupyter lab jcs/notebooks/` and select its **Python 3** kernel. Run the cells in order. The companion notebooks are [Model and mathematics](01_model_and_mathematics.ipynb) and [Exploring saved worlds](02_exploring_saved_worlds.ipynb).

In [ ]:
from pathlib import Path
import sys
from dataclasses import replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML

%matplotlib inline

here = Path.cwd().resolve()
JCS_ROOT = next((p for p in [here, *here.parents] if (p / "src/human_ai_commons").is_dir()), None)
if JCS_ROOT is None and (here / "jcs/src/human_ai_commons").is_dir():
    JCS_ROOT = here / "jcs"
if JCS_ROOT is None:
    raise RuntimeError("Open the notebook from the JCS folder or its parent repository.")
for path in [JCS_ROOT / "src", JCS_ROOT / "notebooks"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from _explore import style, BLUE, TEAL, ORANGE, GRAY, frame_map, map_legend
from human_ai_commons.paper_experiments import load_protocol, cell_context, design_setup, run_segment, CLASSES
from human_ai_commons.payoffs import Policy
style()
display(HTML("""<style>
.jcs-title {border-left:5px solid #278477;padding:12px 22px;background:#f3f7f7;margin:16px 0 24px}
.jcs-title h1 {margin:4px 0 12px;font-weight:600;color:#244867}
.jcs-kicker {font-size:12px;letter-spacing:2px;color:#278477;text-transform:uppercase}
.jcs-note {border-left:3px solid #bc703a;padding:10px 16px;background:#fbf7f2;margin:16px 0}
</style>"""))

## 1 · Load decides which sources empty

Each agent draws from one source within reach per step, chosen at random. Spreading an agent's lower request $c_i$ evenly over the set $S_i$ of sources it reaches gives the **expected load** of source $j$,

$$\lambda_j=\frac{1}{b_j}\sum_{i:\,j\in S_i}\frac{c_i}{|S_i|},$$

the lower requests that reach the source per step, relative to its baseline renewal $b_j$. The load follows from positions and requests alone, before anything is simulated.

- A source is **underused** when even higher requests stay within its baseline renewal, $1.5\,\lambda_j\le1$.
- It is **overloaded** when even lower requests exceed the largest renewal it can reach, $\lambda_j>m_{\max}$, which is 1.25 at $\theta=3$.
- It is **contested** otherwise. Only at a contested source can the agents' choices change the stock.

The next cell runs the original game on world 0 for 2,000 steps and compares each source's stock over steps 1,500 to 2,000 with its load.

In [ ]:
config = load_protocol(JCS_ROOT / "configs/local_measures.json")
WORLD = 0
CLASS_COLORS = {"underused": "#9cc5e3", "contested": "#e8b33a", "overloaded": BLUE}

world, reference, requests, drivers = cell_context(config, "reference", WORLD)
made, physical, options = design_setup(config, "unchanged", reference)
_, _, late, info = run_segment(world, physical, made, drivers, 0, 2000, cooperative_request=requests,
                               windows=[("late", 1500, 2000)], **options)
late_stock = late["late"]["source_stock"]
load, classes = info["load"], info["classes"]
reached = classes >= 0

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), layout="constrained")
field = np.full((world.spec.height, world.spec.width, 3), .95)
for code_, name in enumerate(CLASSES):
    color = np.array([int(CLASS_COLORS[name][k:k + 2], 16) / 255 for k in (1, 3, 5)])
    for x, y in world.resources[classes == code_]:
        field[y, x] = color
axes[0].imshow(field, origin="lower", extent=(-.5, world.spec.width - .5, -.5, world.spec.height - .5))
for kind, marker in [(False, "o"), (True, "^")]:
    mask = world.kinds == kind
    axes[0].scatter(world.positions[mask, 0], world.positions[mask, 1], marker=marker, s=22, color="white", edgecolor="black", linewidth=.5)
axes[0].set(title="Load classes in world 0", xlabel="Grid x", ylabel="Grid y",
            xticks=range(0, world.spec.width, 2), yticks=range(0, world.spec.height, 2))
for code_, name in enumerate(CLASSES):
    member = classes == code_
    axes[1].scatter(np.minimum(load[member], 3), 100 * late_stock[member], s=22, color=CLASS_COLORS[name], label=name)
axes[1].axhline(10, color=GRAY, lw=.8, ls=":")
axes[1].text(2.95, 12, "nearly empty below this line", ha="right", fontsize=9, color=GRAY)
axes[1].set(xlabel="Expected load (loads of 3 or more drawn at 3)", ylabel="Stock in steps 1,500 to 2,000 (% of capacity)",
            title="Stock against load", ylim=(-3, 103))
axes[1].legend(frameon=False, title="source")
plt.show(); plt.close(fig)

summary = pd.DataFrame({
    "Sources": [int((classes == k).sum()) for k in range(3)],
    "Mean stock (% of capacity)": [100 * late_stock[classes == k].mean() for k in range(3)],
    "Nearly empty (%)": [100 * (late_stock[classes == k] < .1).mean() for k in range(3)],
}, index=list(CLASSES)).round(1)
display(summary)

The stock is ordered by the load. Overloaded sources stay nearly empty under any choice between the two requests, underused sources stay close to full, and only the contested sources lie in between. This is why the paper finds that changing the payoffs, which changes only how much each agent requests, cannot lift the depleted sources.

## 2 · Three interventions on the AI agents

The interventions bind the AI agents only, because they account for 95% of what is requested. Human agents keep choosing a source at random and are never restricted, and the payoffs stay those of the original game.

- **Scarcity pricing.** The model contains no money, so a price that rises as a source empties is represented by its effect: an AI agent chooses source $j$ among those it reaches with probability proportional to $e^{\kappa n_j}$, with $\kappa=4$.
- **Extraction cap.** AI agents may not draw a source below a floor of 25% of its capacity; their requests are cut back in proportion. Human agents may still draw below the floor.
- **Capacity-based siting.** Before the simulation, the AI agents are placed one after another where every source within reach stays within its baseline renewal, preferring cells that share few sources with human agents.

As in the paper, pricing and the cap start after 1,000 steps of the original game, and siting holds from the first step. All are measured over the following 1,500 steps.

In [ ]:
DEVELOP, AFTER = 1000, 1500
developed = {}

def developed_state(cell):
    """The state after 1,000 steps of the original game, computed once per placement."""
    if cell not in developed:
        w, scen, req, seed = cell_context(config, cell, WORLD)
        made, physical, options = design_setup(config, "unchanged", scen)
        state, *_ = run_segment(w, physical, made, seed, 0, DEVELOP, cooperative_request=req, **options)
        developed[cell] = (w, scen, req, seed, state)
    return developed[cell]

def measure(cell, physical_of):
    w, scen, req, seed, state = developed_state(cell)
    made, physical, options = physical_of(scen)
    final, rows, _, _ = run_segment(w, physical, made, seed, DEVELOP, DEVELOP + AFTER, state=state,
                                    cooperative_request=req, windows=[("after", DEVELOP, DEVELOP + AFTER)], **options)
    row = rows[0]
    return dict(human=100 * row["human_activity"], ai=100 * row["ai_activity"], empty=100 * row["empty_share"]), final, w

conditions = {
    "Original game": ("reference", "unchanged"),
    "Scarcity pricing": ("reference", "price_4"),
    "Extraction cap": ("reference", "cap_25"),
    "Pricing with the cap": ("reference", "price_4_cap_25"),
    "Capacity-based siting": ("sited_all", "unchanged"),
    "Siting with pricing": ("sited_all", "price_4"),
}
results, finals = {}, {}
for label, (cell, name) in conditions.items():
    results[label], finals[label], _ = measure(cell, lambda scen, name=name: design_setup(config, name, scen))
table = pd.DataFrame(results).T.round(1)
table.columns = ["Human agents: share of demand met (%)", "AI agents: share of demand met (%)", "Sources nearly empty (%)"]
display(table)

fig, ax = plt.subplots(figsize=(11, 3.8), layout="constrained")
position = np.arange(len(table))
ax.bar(position - .2, table.iloc[:, 0], .38, color=TEAL, label="Human agents")
ax.bar(position + .2, table.iloc[:, 1], .38, color=BLUE, label="AI agents")
ax.set(xticks=position, ylabel="Share of demand met (%)", ylim=(0, 100), title="World 0, 1,500 steps after the change")
ax.set_xticklabels(table.index, fontsize=9)
ax.legend(frameon=False, loc="upper left")
plt.show(); plt.close(fig)

saved = JCS_ROOT / "results/local_measures/rows.jsonl"
if saved.is_file():
    import json
    stored = [json.loads(line) for line in saved.read_text().splitlines()]
    def stored_row(cell, name):
        return next(r for r in stored if r["world_index"] == WORLD and r["cell"] == cell and r["condition"] == name
                    and r.get("timing") == "after_development" and r.get("window") == "0-1500")
    same = all(np.isclose(results[label]["human"], 100 * stored_row(c, n)["human_activity"]) for label, (c, n) in conditions.items())
    print("Agrees with the saved experiment for world 0." if same else "Differs from the saved experiment; check the configuration.")

<div class="jcs-note"><strong>Reading the table.</strong> Pricing serves the AI agents: they move to fuller sources. The cap serves the human agents first, because what lies below the floor is in effect reserved for them, and it keeps every source above empty, where renewal can rise above its baseline rate. Siting raises the limit on what can be delivered, so both groups gain. World 0 is poorer than the average world, so its levels are lower than the paper's averages; the direction of each change is the same.</div>

### Vary the strength

Choose a price strength $\kappa$ (0 switches pricing off), a floor for the cap (0 switches it off), and whether the AI agents are sited. Each change reruns the 1,500 steps after development, which takes about a second; the developed states are reused.

In [ ]:
def custom(kappa, floor):
    def physical_of(scen):
        changed = scen
        if kappa > 0:
            changed = replace(changed, selection="priced", price=float(kappa), regulated="ai")
        if floor > 0:
            changed = replace(changed, floor=float(floor), regulated="ai")
        return Policy(), changed, {}
    return physical_of

def show_strength(kappa=4., floor=.25, sited=False):
    cell = "sited_all" if sited else "reference"
    outcome, final, w = measure(cell, custom(kappa, floor))
    base = results["Capacity-based siting" if sited else "Original game"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.9), layout="constrained", gridspec_kw={"width_ratios": [1, 1.25]})
    position = np.arange(3)
    axes[0].bar(position - .2, [base["human"], base["ai"], base["empty"]], .38, color=GRAY, label="Without pricing or cap")
    axes[0].bar(position + .2, [outcome["human"], outcome["ai"], outcome["empty"]], .38, color=TEAL, label="With these settings")
    axes[0].set(xticks=position, ylim=(0, 100), ylabel="%", title="World 0, 1,500 steps after the change")
    axes[0].set_xticklabels(["Human agents:\nshare of demand met", "AI agents:\nshare of demand met", "Sources\nnearly empty"], fontsize=9)
    axes[0].legend(frameon=False, fontsize=8)
    image = frame_map(axes[1], w, final.water, final.strategies, "Stock at the last step")
    fig.colorbar(image, ax=axes[1], shrink=.75, label="Stock / capacity")
    fig.legend(handles=map_legend(), loc="outside lower center", ncol=4, frameon=False, fontsize=9)
    plt.show(); plt.close(fig)

kappa = widgets.FloatSlider(value=4., min=0., max=8., step=.5, description="Price κ", continuous_update=False)
floor = widgets.FloatSlider(value=.25, min=0., max=.6, step=.05, description="Cap floor", continuous_update=False)
sited = widgets.Checkbox(value=False, description="AI agents sited")
display(widgets.HBox([kappa, floor, sited]), widgets.interactive_output(show_strength, {"kappa": kappa, "floor": floor, "sited": sited}))

A higher floor reserves more of each source for the human agents and, beyond a point, costs the AI agents. A stronger price adds little once the AI agents already find the fuller sources. These are the trade-offs that the paper measures across 64 worlds and seven settings.

## 3 · From one world to the paper

The paper's numbers are means over 64 worlds, compared within each world. When the interventions experiment has been run, `.venv/bin/python jcs/reproduce.py paper all local_measures`, the next cell reads its tables and shows the reference setting.

In [ ]:
report = JCS_ROOT / "results/local_measures/report"
if (report / "intervention_effects.csv").is_file():
    effects = pd.read_csv(report / "intervention_effects.csv")
    effects = effects[(effects.cell == "reference") & (effects.timing == "after_development") & (effects.window == "0-1500")]
    placement = pd.read_csv(report / "intervention_placement_effects.csv")
    placement = placement[(placement.window == "0-1500") & (placement.cell == "sited_all")]
    names = {"price_4": "Scarcity pricing", "cap_25": "Extraction cap", "price_4_cap_25": "Pricing with the cap"}
    rows = []
    for design, label in names.items():
        r = effects[effects.design == design].iloc[0]
        rows.append((label, r.human_activity_unchanged, r.human_activity_difference_mean, r.ai_activity_unchanged, r.ai_activity_difference_mean))
    r = placement.iloc[0]
    rows.append(("Capacity-based siting", r.human_activity_random_placement, r.human_activity_difference_mean, r.ai_activity_random_placement, r.ai_activity_difference_mean))
    paper = pd.DataFrame([(label, 100 * h0, 100 * (h0 + dh), 100 * a0, 100 * (a0 + da)) for label, h0, dh, a0, da in rows],
                         columns=["Intervention", "Human agents: before (%)", "Human agents: after (%)", "AI agents: before (%)", "AI agents: after (%)"]).set_index("Intervention").round(0)
    display(Markdown("**Reference setting, mean over 64 worlds**"), paper)
else:
    display(Markdown("The experiment's tables are not present. Run `.venv/bin/python jcs/reproduce.py paper all local_measures` from the repository root to generate them."))

| What this notebook shows | What it does not establish |
|---|---|
| How load sorts the sources of one world | That every world looks the same; the paper compares 64 |
| What each intervention changes in that world | Statistical support; the paper's intervals come from paired comparisons over worlds |
| How the strength of pricing and the cap shifts the balance between the groups | Costs, compliance or how a real authority would set a price or a floor |

The definitions and the full set of settings are in the [experiment descriptions](../docs/experiments.md).